Jacob Maurer \
6/2/2026 \
Purpose: To demo a possible chemical representation for the KAN networks. Use the images of the molecules, then preprocess. This script demonstrates this process

### Pipeline Overview
1. Load the dataset using the `open_hopv15_dataset` function
2. Convert the loaded data with the chem `MolFromSmiles`
3. Add hs using the chem `AddHs` function
4. Convert molecules to images using rdkit's Draw class `MolToImage`
5. Convert the image to gray scale with PIL's `ImageOps.grayscale`
6. Flatten the image to get a single long array
7. Standardize the data using scikit-learn's `StandardScaler`
8. Apply PCA with scikit learn (use .9 at the threshold)
9. add any other variables we wish to include (ex. HOMO, LUMO, etc...)

In [2]:
from import_dataset import open_hopv15_dataset, mol_atom_dist
from rdkit import Chem
from rdkit.Chem import Draw
import matplotlib.pyplot as plt

data = open_hopv15_dataset("./HOPV_15_revised_2.data")
chems = [Chem.AddHs(Chem.MolFromSmiles(molecule.smiles_molecule)) for molecule in data if not molecule.experimental_data.has_nan()]

In [3]:
import numpy as np
from PIL import ImageOps
imgs = np.array([np.array(ImageOps.grayscale(Draw.MolToImage(chem))).flatten() for chem in chems])

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
standard_imgs = scaler.fit_transform(imgs)
pca = PCA(n_components=.9)
pca_imgs = pca.fit_transform(standard_imgs) 

In [38]:
pca_imgs

array([[-24.82033182, -11.03102831,  -4.87244894, ...,   0.40437184,
          0.69267964,  -0.48630674],
       [  7.49368385,  -7.03881779,  -4.94798908, ...,  -2.98820678,
        -10.10086142,  -0.84704531],
       [ 18.32652104,  -4.27610908,   6.59971569, ...,  16.3069333 ,
         15.25846364,  -1.35752776],
       ...,
       [ 24.94843984,  -5.82776295,   7.95059475, ...,  -1.02938436,
         -3.05758898,  -2.46056143],
       [ 20.11467884,  -5.44508838,   6.46487776, ..., -22.36439918,
        -23.90803369,  18.17600266],
       [ 14.30733568,  -8.05478546,  -2.0483104 , ...,   6.18628092,
        -13.47091706,   5.77767061]], shape=(350, 259))

In [17]:
pca.explained_variance_ratio_[:218].sum()

np.float64(0.9010292174795242)

In [22]:
pca.components_[0]

array([-4.75945695e-17,  1.79226878e-19,  3.11818239e-17, ...,
       -0.00000000e+00, -0.00000000e+00, -0.00000000e+00], shape=(90000,))

In [7]:
# Created 6/11/2026
import random

from molecule_autoencoder import MoleculeAutoEncoderSigmoid, create_autoencoder, create_layers, save_model, train, test, CoordinateDataset, MoleculeAutoEncoderTanh
import torch.nn as nn
import torch
from torch.utils.data import DataLoader

device = "cuda"
test_size = 0.2
train_set, test_set = imgs[int(test_size*len(imgs)):], imgs[:int(test_size*len(imgs))]
train_loader = DataLoader(CoordinateDataset(np.asarray(train_set)), batch_size=100, shuffle=True)
test_loader = DataLoader(CoordinateDataset(np.asarray(test_set)), batch_size=100, shuffle=True)
layer_strs = create_autoencoder(90000, 100, 4, ["tanh"], (500, 800))
encoder_layers= create_layers(layer_strs[0], {"relu": nn.ReLU(), "silu": nn.SiLU(), "tanh": nn.Tanh(), "sig": nn.Sigmoid()})
decoder_layers= create_layers(layer_strs[1], {"relu": nn.ReLU(), "silu": nn.SiLU(), "tanh": nn.Tanh(), "sig": nn.Sigmoid()})
print(f"Encoder: {layer_strs[0]}")
print(f"Decoder: {layer_strs[1]}")
limit = True if random.random() > 0.5 else False
print(f"Clamp enforced: {limit}")
model = MoleculeAutoEncoderTanh(encoder_layers, decoder_layers, (0, 255) if limit else None).to(device)
loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
max_epochs = 100
min_MSE = 1000
train_loss = 20000
test_loss = 20000
curr_epoch = 0
while test_loss > min_MSE:
    print(f"Epoch: {curr_epoch}")
    if max_epochs == curr_epoch:
        break
    train_loss = train(train_loader, model, loss, optimizer)
    test_loss = test(test_loader, model, loss)
    curr_epoch += 1
if max_epochs == curr_epoch:
    print(f"This parameter set could not acheive an MSE of {min_MSE} within {max_epochs} epochs. This would use too many resources, rerun to pursue a different path.")
else:
    print(f"Found a decent set of parameters! Make sure to save these, then continue training later if possible.")

Encoder: 90000|759->tanh->759|671->tanh->671|512->tanh->512|530->tanh->530|521->tanh->521|100
Decoder: 100|728->tanh->728|680->tanh->680|526->tanh->526|685->tanh->685|777->tanh->777|90000
Clamp enforced: True
Epoch: 0
Avg loss: 62043.949219 

Epoch: 1
Avg loss: 61264.781250 

Epoch: 2
Avg loss: 60492.089844 

Epoch: 3
Avg loss: 59692.484375 

Epoch: 4
Avg loss: 58883.148438 

Epoch: 5
Avg loss: 58073.800781 

Epoch: 6
Avg loss: 57269.839844 

Epoch: 7
Avg loss: 56474.320312 

Epoch: 8
Avg loss: 55689.183594 

Epoch: 9
Avg loss: 54915.421875 

Epoch: 10
Avg loss: 54153.769531 

Epoch: 11
Avg loss: 53404.617188 

Epoch: 12
Avg loss: 52668.109375 

Epoch: 13
Avg loss: 51944.308594 

Epoch: 14
Avg loss: 51233.152344 

Epoch: 15
Avg loss: 50534.558594 

Epoch: 16
Avg loss: 49848.378906 

Epoch: 17
Avg loss: 49174.429688 

Epoch: 18
Avg loss: 48512.597656 

Epoch: 19
Avg loss: 47862.574219 

Epoch: 20
Avg loss: 47224.285156 

Epoch: 21
Avg loss: 46597.398438 

Epoch: 22
Avg loss: 45981.86718

In [5]:
model.eval()
with torch.no_grad():
    for batch, (x, y) in enumerate(test_loader):
        model.encoder.to(device)
        model.decoder.to(device)
        for item in model(x):
            print(item)

tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.], device='cuda:0')
tensor([255., 255.,   0.,  ..., 255., 255.,   0.

In [5]:
save_model(model, "./image_model/", "image_data_autoencoder_non_clamp_3",train_loss,test_loss, layer_strs[0], layer_strs[1])